# 08.6f ACE-Step 生成与控制入口

ACE-Step 代表可控生成和后续 LoRA 个性化路线。Notebook 展示两种入口：`acestep --port ...` 的交互界面，以及 `ACEStepPipeline` 的脚本化生成；环境完整时会直接运行脚本化生成。


## 运行环境与安装

在 Jupyter 中选择 ACE-Step 对应 kernel。首次运行前先建立独立环境。mac/Linux 终端使用：

Hugging Face 当前 CLI 命令名为 `hf`。若 `hf --help` 不可用，可按官方文档先安装 standalone CLI；mac/Linux 使用 `curl -LsSf https://hf.co/cli/install.sh | bash`，Windows PowerShell 使用 `powershell -ExecutionPolicy ByPass -c "irm https://hf.co/cli/install.ps1 | iex"`。也可以把下面的 `hf download ...` 改成 `uvx hf download ...`。

```bash
cd CODE
python3.10 -m venv venv_ch08_acestep
source venv_ch08_acestep/bin/activate
python -m pip install --upgrade pip setuptools wheel
mkdir -p external chapter08/models
git clone https://github.com/ace-step/ACE-Step.git external/ACE-Step
python -m pip install -e external/ACE-Step
python -m pip install pandas PyYAML torchcodec ipykernel ipywidgets
python -m ipykernel install --user --name chapter08-acestep --display-name "Python 3.10 (chapter08-acestep)"
hf download ACE-Step/ACE-Step-v1-3.5B --local-dir chapter08/models/ace_step_v1_3_5b
acestep --checkpoint_path chapter08/models/ace_step_v1_3_5b --port 7865 --bf16 false
```

Windows PowerShell 使用：

```powershell
cd CODE
py -3.10 -m venv venv_ch08_acestep
.\venv_ch08_acestep\Scripts\Activate.ps1
python -m pip install --upgrade pip setuptools wheel
New-Item -ItemType Directory -Force external
New-Item -ItemType Directory -Force chapter08\models
git clone https://github.com/ace-step/ACE-Step.git external/ACE-Step
python -m pip install -e external/ACE-Step
python -m pip install pandas PyYAML torchcodec ipykernel ipywidgets
python -m ipykernel install --user --name chapter08-acestep --display-name "Python 3.10 (chapter08-acestep)"
hf download ACE-Step/ACE-Step-v1-3.5B --local-dir chapter08/models/ace_step_v1_3_5b
acestep --checkpoint_path chapter08/models/ace_step_v1_3_5b --port 7865 --bf16 false
```

如果 `git clone` 网络不稳定，可以在浏览器下载 https://github.com/ace-step/ACE-Step 的 ZIP，解压后把源码文件夹移动并命名为 `CODE/external/ACE-Step`，再执行 `python -m pip install -e external/ACE-Step`。

ACE-Step 的 GitHub 仓库只是源码，不包含预训练权重。权重需要单独下载到 `CODE/chapter08/models/ace_step_v1_3_5b`；该目录应直接包含 `music_dcae_f8c8`、`music_vocoder`、`ace_step_transformer`、`umt5-base`。如果不用 `hf download`，也可以从 Hugging Face 页面手动下载 `ACE-Step/ACE-Step-v1-3.5B` 的文件，解压或移动到这个目录。

本 Notebook 会通过 `--checkpoint_path` 或 `ACEStepPipeline(checkpoint_dir=...)` 使用 `chapter08/models/ace_step_v1_3_5b`，不会把源码目录当作模型权重目录。macOS、CPU 或不支持 bf16 的设备使用 `--bf16 false`；CUDA 且支持 bf16 的环境可改成 `--bf16 true`。ACE-Step 内部用 `torchaudio.save` 写出 wav；较新的 torchaudio 会调用 TorchCodec 保存后端，因此环境中需要安装 `torchcodec`。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Audio, display

from _common.config import load_yaml_config
from _common.device_utils import choose_device
from _common.paths import portable_path
from evaluation.comparison_table import append_model_comparison
from model_runners.base import GenerationRequest
from model_runners.conditioning import build_conditioning_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)

def resolve_device(config=None):
    requested = os.getenv("CHAPTER08_DEVICE")
    if requested is None and config is not None:
        requested = str(config.get("device", "auto"))
    return choose_device(requested or "auto")

def print_setup_guidance(status):
    print(status.reason)
    if status.next_action:
        print(status.next_action)
    print("After completing the setup or moving to compatible hardware, rerun this Notebook; it will load and run the model directly.")

def print_runtime_guidance(error):
    print(str(error))
    print("Resolve the message above, then rerun this Notebook or the current cell.")

from model_runners.ace_step import ACEStepRunner

runner = ACEStepRunner()
status = runner.check_environment()
config = load_yaml_config(ROOT / "configs" / "ace_step_inference.yaml")
display(pd.DataFrame([status.as_row()]))


In [ ]:
display(pd.DataFrame(build_conditioning_rows("ace_step")))
condition_row = {
    "prompt": config["prompts"][0]["text"],
    "lyrics": config.get("lyrics", ""),
    "checkpoint_dir": config.get("checkpoint_dir", "models/ace_step_v1_3_5b"),
    "duration_seconds": config.get("duration_seconds", ""),
    "infer_steps": config.get("infer_steps", ""),
    "seed": 8,
}
display(pd.DataFrame([condition_row]))


In [ ]:
print("ACE-Step Gradio command:")
print(
    runner.build_gradio_command(
        port=int(os.getenv("CHAPTER08_ACE_STEP_PORT", "7865")),
        checkpoint_dir=ROOT / config.get("checkpoint_dir", "models/ace_step_v1_3_5b"),
        bf16=bool(config.get("bf16", False)),
    ).shell_command()
)

prompt = config["prompts"][0]
request = GenerationRequest(
    prompt=prompt["text"],
    prompt_id=prompt["prompt_id"],
    duration_sec=float(config.get("duration_seconds", 20)),
    output_dir=ROOT / config["outputs"]["audio_dir"],
    seed=8,
    extra={
        "device": resolve_device(config),
        "checkpoint_dir": ROOT / config.get("checkpoint_dir", "models/ace_step_v1_3_5b"),
        "dtype": config.get("dtype", "float32"),
        "lyrics": config.get("lyrics", ""),
        "infer_steps": int(config.get("infer_steps", 60)),
    },
)

if status.available:
    try:
        result = runner.generate(request)
    except RuntimeError as exc:
        print_runtime_guidance(exc)
    else:
        append_model_comparison(
            ROOT / config["outputs"]["table_csv"],
            {
                "model_name": result.model_name,
                "prompt_id": result.prompt_id,
                "dataset_context": "style prompt and lyrics",
                "duration_sec": result.duration_sec,
                "wall_time_sec": result.wall_time_sec,
                "device": result.device,
                "output_audio_path": result.output_audio_path,
            },
        )
        display(Audio(str(result.output_audio_path)))
else:
    print_setup_guidance(status)
